# 🕵️ Dia 71 — Deepfake de voz, vídeo e imagem em fraudes

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 71* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-71.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Segunda-feira, 26/out. Seu Antônio, 71 anos, cooperado da agência Centro, fez dois Pix "autorizados" depois de uma ligação do "Cláudio, gerente da agência". Na noite anterior, ele recebeu um áudio de WhatsApp com a voz do Cláudio. O Cláudio real estava num treinamento, com o celular desligado.

Rafa: *"A pergunta errada é 'esse áudio é falso?'. A certa é: que procedimento teria parado o Seu Antônio mesmo que a voz fosse perfeita?"*

Neste notebook você vai:

1. **Medir um sinal de áudio** (o "limpo demais") e ver por que ele é só de apoio.
2. **Procurar Content Credentials (C2PA) dentro de um JPEG**, byte a byte, e entender por que "sem credenciais" não diz nada.
3. **Jogar o quiz "qual checagem pesa mais?"**.
4. **Gerar o procedimento de retorno de ligação da Serra Azul**, com SHA-256.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo aqui é fictício (Operação Isca).

## 1. O áudio "limpo demais": um sinal de apoio, medido

O checklist do Dia 71 cita *"áudio chapado, estranhamente limpo, ou sem respiração e sem som de ambiente"*. Uma ligação real de celular tem sempre um pouco de ruído de fundo, mesmo nas pausas. Um áudio sintetizado e colado pode ter **silêncio digital absoluto** entre as palavras.

Vamos gerar dois áudios fictícios com a biblioteca `wave` (padrão do Python) e medir o **piso de ruído** quadro a quadro. Atenção: o resultado vai parecer "convincente", e é exatamente isso que você precisa aprender a desconfiar. Aplicativos de mensagem aplicam supressão de ruído, e uma pessoa real gravando num quarto silencioso também produz pausas quase mudas.

In [ ]:
import math      # seno e raiz quadrada para gerar e medir o som
import random    # ruído de sala (aleatório, mas com semente fixa para o resultado ser reproduzível)
import struct    # converte números em bytes no formato que o WAV espera
import wave      # lê e escreve arquivos .wav sem instalar nada

TAXA = 8000      # 8 kHz: qualidade de ligação telefônica
random.seed(71)  # semente fixa: todo aluno vê os mesmos números

def gerar_audio(caminho, com_ruido_de_sala):
    """Gera 3 s de 'fala' falsa: rajadas de tom (as 'palavras') separadas por pausas."""
    amostras = []
    for n in range(TAXA * 3):                     # 3 segundos de amostras
        t = n / TAXA                              # tempo em segundos
        falando = int(t * 2) % 2 == 0             # alterna 0,5 s falando / 0,5 s em pausa
        valor = 0.5 * math.sin(2 * math.pi * 220 * t) if falando else 0.0
        if com_ruido_de_sala:
            valor += random.gauss(0, 0.01)        # chiado leve, presente o tempo todo
        valor = max(-1.0, min(1.0, valor))        # evita estourar o limite do formato
        amostras.append(int(valor * 32767))       # 16 bits com sinal
    with wave.open(caminho, "wb") as w:
        w.setnchannels(1)                         # mono, como uma ligação
        w.setsampwidth(2)                         # 2 bytes = 16 bits por amostra
        w.setframerate(TAXA)
        w.writeframes(struct.pack(f"<{len(amostras)}h", *amostras))

def piso_de_ruido(caminho, janela_ms=20):
    """Devolve a fração de janelas de 20 ms que estão em silêncio digital quase absoluto."""
    with wave.open(caminho, "rb") as w:
        bruto = w.readframes(w.getnframes())
    amostras = struct.unpack(f"<{len(bruto) // 2}h", bruto)
    passo = TAXA * janela_ms // 1000             # 160 amostras por janela de 20 ms
    mudas = total = 0
    for i in range(0, len(amostras) - passo, passo):
        bloco = amostras[i:i + passo]
        rms = math.sqrt(sum(a * a for a in bloco) / passo) / 32767   # energia média da janela
        total += 1
        mudas += rms < 0.001                      # abaixo disso, nenhum microfone real fica
    return mudas / total

gerar_audio("ligacao_com_sala.wav", com_ruido_de_sala=True)
gerar_audio("audio_limpo_demais.wav", com_ruido_de_sala=False)

for nome in ("ligacao_com_sala.wav", "audio_limpo_demais.wav"):
    print(f"{nome:28s} janelas em silêncio digital: {piso_de_ruido(nome):.0%}")

> 💡 **Repare no que você acabou de ver.** O número separa os dois arquivos com perfeição, e isso é uma armadilha. No mundo real: (1) o WhatsApp e muitos celulares aplicam supressão de ruído, que zera as pausas de áudios legítimos; (2) um bom gerador acrescenta ruído de sala de propósito. Por isso o original coloca esse sinal no checklist de alertas, mas nunca como a checagem que decide. No registro de verificação, ele entra na linha "Visual/escuta", marcado como **apoio**.

## 2. Content Credentials (C2PA) dentro de um JPEG

Num JPEG, o manifesto C2PA fica em segmentos **APP11** (marcador `FF EB`), dentro de caixas JUMBF rotuladas `c2pa`. O cabeçalho do JPEG é uma fila de segmentos: `FF` + tipo + tamanho (2 bytes) + conteúdo. Dá para percorrer essa fila só com bytes.

Abaixo montamos **dois JPEGs mínimos fictícios** (não são imagens visíveis, só a estrutura): um com um segmento APP11 que contém `c2pa`, outro sem. Se você tiver uma foto sua com Content Credentials, suba no Colab e troque o caminho.

In [ ]:
from pathlib import Path

def segmento(marcador, conteudo):
    """Monta um segmento JPEG: FF + marcador + tamanho (inclui os 2 bytes do próprio tamanho) + conteúdo."""
    return bytes([0xFF, marcador]) + (len(conteudo) + 2).to_bytes(2, "big") + conteudo

SOI = b"\xff\xd8"                                   # início da imagem
EOI = b"\xff\xd9"                                   # fim da imagem
app0 = segmento(0xE0, b"JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00")   # cabeçalho JFIF comum
# APP11 fictício: 'JP' + caixa JUMBF com o rótulo 'c2pa' (estrutura simplificada, só para o exercício)
app11 = segmento(0xEB, b"JP\x00\x01\x00\x00\x00\x01" + b"\x00\x00\x00\x20jumb" + b"\x00\x00\x00\x18jumdc2pa\x00manifesto-ficticio")
sos = segmento(0xDA, b"\x00" * 10) + b"\x12\x34" * 8   # início dos dados comprimidos (lixo fictício)

Path("com_credenciais.jpg").write_bytes(SOI + app0 + app11 + sos + EOI)
Path("sem_credenciais.jpg").write_bytes(SOI + app0 + sos + EOI)

def tem_manifesto_c2pa(caminho):
    """True se existir um APP11 com a etiqueta 'c2pa'. Não valida a assinatura!"""
    dados = Path(caminho).read_bytes()
    if dados[:2] != b"\xff\xd8":                    # não começa com SOI: não é JPEG
        return False
    i = 2
    while i + 4 <= len(dados) and dados[i] == 0xFF:
        marcador = dados[i + 1]
        tamanho = int.from_bytes(dados[i + 2:i + 4], "big")
        if marcador == 0xEB and b"c2pa" in dados[i + 4:i + 2 + tamanho]:
            return True
        if marcador == 0xDA:                          # SOS: daqui em diante é imagem comprimida
            break
        i += 2 + tamanho                              # pula para o próximo segmento
    return False

for nome in ("com_credenciais.jpg", "sem_credenciais.jpg"):
    achou = tem_manifesto_c2pa(nome)
    leitura = ("manifesto presente: valide a assinatura no Content Credentials Verify ou com c2patool"
               if achou else "sem credenciais: NÃO é evidência para nenhum lado")
    print(f"{nome:22s} → {leitura}")

> ⚖️ **Por que ler localmente?** Num caso real, o arquivo pode conter dado pessoal (o rosto, a voz, o número de alguém). Antes de mandar para um site de terceiros, saiba se vale a pena e se o seu escopo permite. Ler os bytes não envia nada a ninguém.

## 3. 🎮 Quiz: qual checagem pesa mais?

Para cada situação, escolha a checagem que deve **mais pesar** na conclusão, ou o passo que resolve o problema. Itens 🇺🇸 vêm do original; itens 🇧🇷, da Operação Isca.

Quer só ler as respostas? Mude `MODO_JOGO` para `False`.

In [ ]:
MODO_JOGO = True   # False = só mostra o gabarito

def perguntar(texto, validas, tentativas=5):
    """Pede uma resposta até ela ser válida (com limite, para nunca travar)."""
    for _ in range(tentativas):
        r = input(texto).strip().lower()
        if r in validas:
            return r
        print("   Responda com uma das opções:", ", ".join(validas))
    return ""        # desistiu: conta como erro

# (bandeira, situação, opções, letra certa, explicação)
QUIZ = [
    ("🇺🇸", "Vídeo do 'ministro' recomendando quantum-yield.example. O que mais pesa?",
     {"a": "Lábios fora de sincronia a partir de 0:12", "b": "Quadro-chave igual a uma entrevista de 2023 com outro áudio", "c": "Detector deu 92% de chance de deepfake"},
     "b", "Uma contradição com um original verificável. Os artefatos visuais e o detector são só apoio."),
    ("🇺🇸", "O Content Credentials Verify diz 'no credentials'. Isso significa:",
     {"a": "Que o arquivo é falso", "b": "Que o arquivo é autêntico", "c": "Quase nada: a maioria das plataformas remove as credenciais"},
     "c", "Ausência de credenciais não é evidência para nenhum lado."),
    ("🇺🇸", "Um detector diz 'nenhuma manipulação detectada'. Você conclui:",
     {"a": "Muito pouco: detectores ficam para trás dos geradores", "b": "Autêntico, com confiança alta", "c": "Que precisa de um segundo detector para confirmar"},
     "a", "O resultado negativo de um detector vale pouco."),
    ("🇺🇸", "Caso de Hong Kong (videochamada convincente). Qual passo para a transferência?",
     {"a": "Treinar o funcionário para ver deepfakes", "b": "Retorno de ligação para um número do cadastro + segundo aprovador", "c": "Pedir à pessoa no vídeo que vire de lado"},
     "b", "O passo não pode depender de alguém perceber a falsificação."),
    ("🇺🇸", "Quem liga diz 'não desliga, vou perder o sinal'. Pelo procedimento:",
     {"a": "Abre-se exceção pela urgência", "b": "Pede um código por SMS para confirmar", "c": "Mesmo caminho: encerra e liga de volta para o número conhecido"},
     "c", "Objeção e urgência seguem o mesmo caminho, sem exceções."),
    ("🇧🇷", "Áudio do 'Cláudio' para o Seu Antônio. O que mais pesa?",
     {"a": "Pausas em silêncio digital absoluto", "b": "O Cláudio real estava com o celular desligado e o número não está em nenhum cadastro", "c": "A voz 'parece um pouco robótica'"},
     "b", "Um fato verificável que contradiz a mídia vale mais que qualquer escuta."),
    ("🇧🇷", "O áudio veio por WhatsApp e não tem Content Credentials. Você escreve:",
     {"a": "'Sem credenciais' e segue, sem tratar como evidência", "b": "'Indício de falsificação'", "c": "'Arquivo adulterado'"},
     "a", "Aplicativos de mensagem não carregam C2PA; a ausência é o normal."),
    ("🇧🇷", "Qual frase o procedimento para cooperados deve trazer?",
     {"a": "'Desconfie se a voz do gerente parecer estranha'", "b": "'Desligue e ligue para o número do seu cartão, mesmo que a voz seja conhecida'", "c": "'Peça ao gerente para dizer seu CPF'"},
     "b", "Regra que vale para todos, inclusive para o gerente verdadeiro; ninguém precisa julgar a voz."),
    ("🇧🇷", "Pedido de 'transferir para uma conta de segurança da cooperativa'. Isso é:",
     {"a": "Procedimento normal em caso de invasão", "b": "Aceitável se o gerente confirmar por áudio", "c": "Mecanismo de golpe: a instituição real não pede isso"},
     "c", "Conta de segurança não existe; é o padrão da falsa central."),
]

acertos = 0
for i, (bandeira, situacao, opcoes, certa, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {bandeira} {situacao}")
    for letra, texto in opcoes.items():           # mostra as alternativas
        print(f"   {letra}) {texto}")
    if MODO_JOGO:
        r = perguntar("   Sua resposta (a/b/c): ", list(opcoes))
        ok = (r == certa)
        acertos += ok
        print("   ✅ Isso!" if ok else f"   ❌ Era {certa}.")
    print("   Por quê:", porque)

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)} — meta do Checkpoint BR: 7.")

### 🧠 Camada 8 — leia antes de seguir

Repare que nenhuma resposta certa depende de **ouvir melhor**. A voz conhecida mantém a vítima no "padrão de verdade" (Levine); a autoridade do gerente e a urgência (Cialdini) fazem o resto. O procedimento vence porque não pede a ninguém que duvide de uma pessoa: pede que todos sigam a mesma regra.

## 4. 📝 O procedimento de retorno de ligação da Serra Azul

Preencha o formulário e rode. Sai `notas/dia-71-procedimento-retorno.md`, no esqueleto do original (gatilho, retorno, aprovação, objeção, palavra-código), com SHA-256.

In [ ]:
#@title Procedimento de retorno de ligação { display-mode: "form" }
organizacao = "Cooperativa Serra Azul (fictícia)"  #@param {type:"string"}
gatilho = "Qualquer pedido, por voz, vídeo, áudio, SMS ou e-mail, que movimente dinheiro, altere dados de pagamento ou peça senha/código"  #@param {type:"string"}
onde_ficam_os_numeros = "Cadastro interno de funcionários e número oficial impresso no cartão do cooperado"  #@param {type:"string"}
limite_segundo_aprovador = "R$ 2.000,00"  #@param {type:"string"}
segundo_aprovador = "Gerente de risco (Marta Quintão) ou substituto formal"  #@param {type:"string"}
frase_objecao = "Urgência, 'não desliga', 'vou perder o sinal': mesmo caminho, sem exceção"  #@param {type:"string"}
palavra_codigo = "Combine com a família uma palavra ou frase secreta; peça-a em qualquer pedido de dinheiro por telefone ou áudio"  #@param {type:"string"}

import hashlib
from datetime import datetime, timezone
from pathlib import Path

agora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
texto = f"""# Procedimento de retorno de ligação — {organizacao}

Versão gerada em {agora} (UTC).

## 1. Gatilho
{gatilho}.

## 2. Pausa
Encerre a ligação ou pause a conversa. Não use nenhum número, link ou contato que veio no pedido.

## 3. Retorno de ligação
Ligue para um número que já estava cadastrado ANTES do pedido chegar. Onde ficam: {onde_ficam_os_numeros}.

## 4. Confirmação
Se a pessoa no número conhecido não confirmar: pare, não pague, escale e preserve o pedido (áudio, prints, número de origem).

## 5. Segundo aprovador
Acima de {limite_segundo_aprovador}: {segundo_aprovador} assina antes de prosseguir.

## 6. Objeção e urgência
{frase_objecao}.

## 7. Palavra-código da família (para funcionários e cooperados)
{palavra_codigo}.

> Nenhum passo deste procedimento pede a alguém que julgue se a voz ou o rosto eram reais.
"""

Path("notas").mkdir(exist_ok=True)
saida = Path("notas/dia-71-procedimento-retorno.md")
saida.write_text(texto, encoding="utf-8")
print(texto)
print("💾", saida, "| SHA-256:", hashlib.sha256(saida.read_bytes()).hexdigest())

try:
    from google.colab import files               # só existe dentro do Colab
    files.download(str(saida))
except ImportError:
    print("(Fora do Colab: o arquivo ficou em notas/)")

## ✅ Antes de fechar o Dia 71

- [ ] Quiz com pelo menos 7/9
- [ ] Registro de verificação dos três itens publicados (Parte 1) e do áudio do "Cláudio" (Prática BR)
- [ ] `notas/dia-71-procedimento-retorno.md` gerado e testado contra Hong Kong e contra o Seu Antônio
- [ ] Procedimento de uma página para cooperados, sem a palavra "deepfake"

**Amanhã (Dia 72):** a Marta encaminha 12 SMS e mensagens de WhatsApp da campanha. Alguns têm cara de ChatGPT. Rafa vai perguntar: *"e daí?"*

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>